<div style="background: linear-gradient(135deg, #0f172a 0%, #1e40af 100%); color: white; padding: 32px 40px; border-radius: 14px;"><div style="font-size: 0.85em; letter-spacing: 0.12em; text-transform: uppercase; opacity: 0.8;">Projet DataViz &mdash; Étape 2</div><div style="font-size: 1.9em; font-weight: 800; margin-top: 6px;">Nettoyage &amp; jointures</div><div style="font-size: 1.05em; margin-top: 10px; opacity: 0.92;">Construction du dataset consolidé <b>Île-de-France</b> &mdash; une ligne par commune, prêt pour l’EDA et le score de tension.</div></div>

## Objectif

Partir des sources brutes et produire **un seul tableau propre** : une ligne par commune francilienne, avec son EPCI, sa population, et les indicateurs d’offre médicale.

| Bloc | Source | Granularité |
| --- | --- | --- |
| Socle communes IDF | INSEE — Code Officiel Géographique 2026 | commune |
| EPCI + population | BANATIC (commune ↔ groupement) | commune |
| **KPI 1** — APL généralistes | DREES (APL 2023) | commune |
| **KPI 2** — densité 4 spécialistes | RPPS / DREES (effectifs libéraux) | département |

> Les 4 spécialités retenues sont **les plus tendues en accès** (étude Doctolib × Fondation Jean-Jaurès 2026) : **cardiologie, dermatologie, ophtalmologie, gynécologie**. Elles ne sont pas disponibles à la commune dans l’open data → on les rattache au **département** (diffusé vers les communes pour la vue maire).

In [ ]:
import pandas as pd, numpy as np, os

RAW = "data/raw/"
IDF_DEPS = ["75", "77", "78", "91", "92", "93", "94", "95"]
SPE = ["Cardiologues", "Dermatologues", "Ophtalmologues", "Gynécologues médicaux et obstétriciens"]
SHORT = {"Cardiologues": "cardio", "Dermatologues": "dermato",
         "Ophtalmologues": "ophtalmo", "Gynécologues médicaux et obstétriciens": "gyneco"}

def num(s):
    """Convertit en nombre en nettoyant espaces (séparateurs de milliers) et virgules décimales."""
    return pd.to_numeric(s.astype(str).str.replace(r"\s+", "", regex=True)
                          .str.replace(",", "."), errors="coerce")

## 1. Socle : les communes d’Île-de-France

On part du Code Officiel Géographique INSEE. On garde les **vraies communes** (`TYPECOM == COM`) de la région Île-de-France (`REG == 11`).

In [ ]:
cog = pd.read_csv(RAW + "insee_v_commune_2026.csv", dtype=str)
idf = cog[(cog.TYPECOM == "COM") & (cog.REG == "11")][["COM", "DEP", "LIBELLE"]].copy()
idf.columns = ["code_insee", "dep", "nom_commune"]
print(f"{len(idf)} communes IDF | départements : {sorted(idf.dep.unique())}")
idf.head()

## 2. EPCI + population (BANATIC)

Le fichier BANATIC relie chaque commune (`insee`) à son intercommunalité (`siren`, nom, nature) et donne sa **population municipale 2025**. C’est la clé de la vue EPCI du dashboard.

In [ ]:
ban = pd.read_csv(RAW + "banatic_commune_siren.csv", sep=";", dtype=str, encoding="latin-1")
ban = ban[ban.dept.isin(IDF_DEPS)][
    ["insee", "siren", "raison_sociale", "nature_juridique", "nb_membres", "pmun_2025"]].copy()
ban.columns = ["code_insee", "epci_siren", "epci_nom", "epci_nature", "epci_nb_communes", "pop_commune"]
ban["pop_commune"] = num(ban["pop_commune"])
print(f"{len(ban)} communes | doublons : {ban.code_insee.duplicated().sum()}")
ban.head()

## 3. KPI 1 — APL généralistes (commune)

L’**Accessibilité Potentielle Localisée** croise l’offre de généralistes et les besoins de la population, par commune. Le fichier DREES a un en-tête sur 9 lignes (lecture en ligne 9).

**Cas Paris** : l’APL est fourni par arrondissement (751xx) et non pour la commune Paris (75056). On agrège les 20 arrondissements en un APL Paris **pondéré par la population standardisée** (méthode préconisée par la DREES).

In [ ]:
apl = pd.read_excel(RAW + "apl_medecins_generalistes_2023.xlsx",
                    sheet_name="APL 2023", header=8, dtype=str)
apl = apl.iloc[1:].copy()  # ligne des unités
apl = apl[["Code commune INSEE", "APL aux médecins généralistes",
           "APL aux médecins généralistes de 60 ans et moins ",
           "Population standardisée 2021 pour la médecine générale"]].copy()
apl.columns = ["code_insee", "apl_generaliste", "apl_gen_moins60", "pop_std"]
for c in ["apl_generaliste", "apl_gen_moins60", "pop_std"]:
    apl[c] = num(apl[c])
apl = apl.dropna(subset=["code_insee"])

# Agrégation des arrondissements parisiens -> 75056
arr = apl[apl.code_insee.str.startswith("751")]
def wmean(g, col):
    w = g["pop_std"]
    return np.average(g[col], weights=w) if w.sum() > 0 else g[col].mean()
paris = pd.DataFrame([{"code_insee": "75056",
                       "apl_generaliste": wmean(arr, "apl_generaliste"),
                       "apl_gen_moins60": wmean(arr, "apl_gen_moins60")}])
apl = pd.concat([apl[~apl.code_insee.str.startswith("751")][["code_insee", "apl_generaliste", "apl_gen_moins60"]],
                 paris], ignore_index=True)
print(f"APL Paris (75056) reconstitué : {paris.apl_generaliste.iloc[0]:.2f}")

## 4. KPI 2 — densité des 4 spécialistes (département)

Les effectifs libéraux sont déclinés par **secteur conventionnel** (secteur 1, secteur 2 / Optam, non conventionnés…). Pour l’effectif total d’une spécialité, on **somme tous les secteurs** de la dernière année disponible, puis on calcule une **densité pour 100 000 habitants**.

In [ ]:
p = pd.read_csv(RAW + "professionnels_sante_liberaux_departement.csv",
                sep=";", dtype=str, encoding="utf-8")
p.columns = [c.replace("\ufeff", "") for c in p.columns]
p["effectif"] = num(p["effectif"])
p["annee"] = p["annee"].astype(int)

ps = p[(p.profession_sante.isin(SPE)) & (p.departement.isin(IDF_DEPS))]
annee = ps.annee.max()
eff = (ps[ps.annee == annee].groupby(["departement", "profession_sante"], as_index=False)["effectif"].sum())
print(f"Effectifs spécialistes — année {annee}")

# population départementale = somme des communes IDF (socle propre)
pop_dep = (idf.merge(ban[["code_insee", "pop_commune"]], on="code_insee", how="left")
             .groupby("dep")["pop_commune"].sum().rename("pop_dep"))

eff = eff.merge(pop_dep, left_on="departement", right_index=True, how="left")
eff["densite_100k"] = eff["effectif"] / eff["pop_dep"] * 100_000
eff["spe"] = eff["profession_sante"].map(SHORT)
dens = eff.pivot(index="departement", columns="spe", values="densite_100k")
dens.columns = [f"densite_{c}_100k" for c in dens.columns]
dens.round(1)

## 5. Assemblage + contrôle qualité

On part du socle des communes IDF et on greffe, par jointure, l’EPCI + population, l’APL, puis la densité spécialistes du département. On vérifie qu’il ne reste **aucun trou**.

In [ ]:
df = (idf
      .merge(ban.drop_duplicates("code_insee"), on="code_insee", how="left")
      .merge(apl, on="code_insee", how="left")
      .merge(dens.reset_index().rename(columns={"departement": "dep"}), on="dep", how="left"))

print("Dimensions :", df.shape)
manquants = df.isna().sum()
print("\nValeurs manquantes :")
print(manquants[manquants > 0] if manquants.any() else '  aucune (OK)')

## 6. Export

Sauvegarde du dataset consolidé (CSV pour Tableau / lecture humaine, Parquet pour la suite Python).

In [ ]:
os.makedirs("data/processed", exist_ok=True)
df.to_csv("data/processed/communes_idf_consolide.csv", index=False, encoding="utf-8-sig")
df.to_parquet("data/processed/communes_idf_consolide.parquet", index=False)
print("[OK] data/processed/communes_idf_consolide.csv  (" + str(len(df)) + " communes)")

## Premiers constats

Le contraste territorial est déjà frappant — il porte tout le projet :

In [ ]:
vue = ["dep", "densite_cardio_100k", "densite_dermato_100k",
       "densite_ophtalmo_100k", "densite_gyneco_100k", "apl_generaliste"]
apercu = df.groupby("dep").agg({"densite_cardio_100k": "first", "densite_dermato_100k": "first",
                                 "densite_ophtalmo_100k": "first", "densite_gyneco_100k": "first",
                                 "apl_generaliste": "mean"}).round(1)
apercu

> **Paris (75)** : dermato ≈ 14,8 / 100k — **Seine-Saint-Denis (93)** : ≈ 1,1 / 100k. Un écart de **plus de 13×** sur une même région. C’est exactement le type de tension que le dashboard doit rendre visible aux élus.

**Étape suivante** → EDA (distributions, corrélations entre KPIs) puis construction du **score de tension**.